# UK Retail Sales Analysis & Forecasting

A compact time-series project using the UK ONS Retail Sales dataset. Selected series: **J467 — seasonally adjusted retail sales volume, excluding fuel**.

**Scope:** sector-level retail sales index, not individual company or product sales.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 1. Load the ONS source file
Upload `drsi.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))

raw = pd.read_csv(file_name, header=None)
print("Raw shape:", raw.shape)


## 2. Select and clean the J467 series
The ONS CSV contains metadata in its first rows. The selected series is in column position 351 (zero-based), with its code shown as J467 in the second metadata row.

In [ ]:
data = raw.iloc[7:, [0, 351]].copy()
data.columns = ["Period", "Retail_Volume_Index"]

data["Period"] = pd.to_datetime(
    data["Period"].astype(str).str.title(),
    format="%Y %b",
    errors="coerce"
)
data["Retail_Volume_Index"] = pd.to_numeric(
    data["Retail_Volume_Index"], errors="coerce"
)

data = (data.dropna()
            .sort_values("Period")
            .loc[lambda x: x["Period"] >= "2021-01-01"]
            .reset_index(drop=True))

print("Cleaned shape:", data.shape)
print("Missing values:", data.isna().sum().to_dict())
display(data.head())
display(data.tail())


## 3. Explore the series

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data["Period"], data["Retail_Volume_Index"])
plt.title("UK Retail Sales Volume Index (excluding fuel)")
plt.xlabel("Month")
plt.ylabel("Index (base year = 100)")
plt.grid(True, alpha=0.3)
plt.show()

print(data["Retail_Volume_Index"].describe())


## 4. Split data chronologically
The final 12 months are held out for testing. Do not randomly shuffle time-series observations.

In [ ]:
ts = data.set_index("Period")["Retail_Volume_Index"].asfreq("MS")
train = ts.iloc[:-12]
test = ts.iloc[-12:]
print("Training observations:", len(train))
print("Testing observations:", len(test))


## 5. Fit Holt-Winters and evaluate

In [ ]:
model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True)

test_forecast = model.forecast(12)
mae = mean_absolute_error(test, test_forecast)
rmse = mean_squared_error(test, test_forecast) ** 0.5

print(f"MAE: {mae:.2f} index points")
print(f"RMSE: {rmse:.2f} index points")

plt.figure(figsize=(12, 5))
plt.plot(train.index, train, label="Training data")
plt.plot(test.index, test, label="Actual test data")
plt.plot(test_forecast.index, test_forecast, label="Forecast")
plt.title("Holt-Winters: 12-month holdout evaluation")
plt.xlabel("Month")
plt.ylabel("Index")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 6. Forecast the next six months
Refit the model using all available observations before generating the final forecast.

In [ ]:
final_model = ExponentialSmoothing(
    ts,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated"
).fit(optimized=True)

future_forecast = final_model.forecast(6)
forecast_df = future_forecast.rename(
    "Forecast_Retail_Volume_Index"
).reset_index()
forecast_df.columns = ["Forecast_Period", "Forecast_Retail_Volume_Index"]
display(forecast_df.round(2))

plt.figure(figsize=(12, 5))
plt.plot(ts.index, ts, label="Historical index")
plt.plot(future_forecast.index, future_forecast, marker="o", label="Six-month forecast")
plt.title("UK Retail Sales Volume: six-month forecast")
plt.xlabel("Month")
plt.ylabel("Index")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 7. Export the outputs

In [ ]:
data.to_csv("UK_Retail_Sales_Cleaned.csv", index=False)
forecast_df.to_csv("UK_Retail_Sales_Forecast.csv", index=False)

from google.colab import files
files.download("UK_Retail_Sales_Cleaned.csv")
files.download("UK_Retail_Sales_Forecast.csv")


## 8. Interpretation and limitations
This is a national sector-level index, not company- or product-level demand. Forecasts are planning estimates and should be interpreted alongside economic conditions and future ONS data revisions.